# HQDeepDTAF — Smoke Test
**Paper:** Jeong et al., *Hybrid quantum neural networks for efficient protein-ligand binding affinity prediction*, EPJ Quantum Technology (2025) 12:120

This notebook runs a 2-epoch smoke test using synthetic data (no PDBbind download needed).
Runtime: ~5-15 min on Colab CPU depending on PennyLane version.

In [ ]:
# Cell 1 — Install dependencies
!pip install torch pennylane numpy tqdm --quiet

In [ ]:
# Cell 2 — Write metrics.py
metrics_src = '''
import numpy as np

def c_index(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    concordant = 0.0
    total = 0
    for i in range(len(y_true)):
        for j in range(i + 1, len(y_true)):
            if y_true[i] == y_true[j]:
                continue
            total += 1
            if y_true[i] < y_true[j]:
                if y_pred[i] < y_pred[j]:
                    concordant += 1
                elif y_pred[i] == y_pred[j]:
                    concordant += 0.5
            else:
                if y_pred[i] > y_pred[j]:
                    concordant += 1
                elif y_pred[i] == y_pred[j]:
                    concordant += 0.5
    return concordant / total if total > 0 else 0.0

def RMSE(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def MAE(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def SD(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    A = np.vstack([y_pred, np.ones(len(y_pred))]).T
    a, b = np.linalg.lstsq(A, y_true, rcond=None)[0]
    residuals = y_true - (a * y_pred + b)
    return float(np.sqrt(np.sum(residuals ** 2) / (len(y_true) - 1)))

def CORR(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.corrcoef(y_true, y_pred)[0, 1])
'''
with open('metrics.py', 'w') as f:
    f.write(metrics_src)
print('metrics.py written')

In [ ]:
# Cell 3 — Write dataset.py
dataset_src = '''
import os
import numpy as np
import torch
from torch.utils.data import Dataset

CHAR_SMI_SET = {
    "(": 1,  ".": 2,  "0": 3,  "2": 4,  "4": 5,  "6": 6,  "8": 7,  "@": 8,
    "B": 9,  "D": 10, "F": 11, "H": 12, "L": 13, "N": 14, "P": 15, "R": 16,
    "T": 17, "V": 18, "Z": 19, "\\\\": 20,"b": 21, "d": 22, "f": 23, "h": 24,
    "l": 25, "n": 26, "r": 27, "t": 28, "#": 29, "%": 30, ")": 31, "+": 32,
    "-": 33, "/": 34, "1": 35, "3": 36, "5": 37, "7": 38, "9": 39, "=": 40,
    "A": 41, "C": 42, "E": 43, "G": 44, "I": 45, "K": 46, "M": 47, "O": 48,
    "S": 49, "U": 50, "W": 51, "Y": 52, "[": 53, "]": 54, "a": 55, "c": 56,
    "e": 57, "g": 58, "i": 59, "m": 60, "o": 61, "s": 62, "u": 63, "y": 64,
}
CHAR_SMI_SET_LEN = len(CHAR_SMI_SET)  # 64

PT_FEATURE_SIZE = 40

AA_TYPES = (\'G\', \'A\', \'V\', \'L\', \'I\', \'M\', \'F\', \'P\', \'W\',
            \'S\', \'T\', \'Y\', \'C\', \'Q\', \'N\', \'D\', \'E\', \'K\', \'R\', \'H\', \'X\')
AA_INDEX = {aa: i for i, aa in enumerate(AA_TYPES)}

SSE_TYPES = (\'B\', \'C\', \'E\', \'G\', \'H\', \'I\', \'S\', \'T\')
SSE_INDEX = {s: i for i, s in enumerate(SSE_TYPES)}

C1_GROUPS = {
    \'non_polar\': frozenset([\'G\', \'A\', \'V\', \'L\', \'I\', \'M\', \'F\', \'P\', \'W\']),
    \'polar\'    : frozenset([\'S\', \'T\', \'Y\', \'C\', \'Q\', \'N\']),
    \'acidic\'   : frozenset([\'D\', \'E\']),
    \'basic\'    : frozenset([\'K\', \'R\', \'H\']),
}
C1_KEYS = (\'non_polar\', \'polar\', \'acidic\', \'basic\')

C2_GROUPS = {
    1: frozenset([\'A\', \'G\', \'V\']),
    2: frozenset([\'I\', \'L\', \'F\', \'P\']),
    3: frozenset([\'Y\', \'M\', \'T\', \'S\']),
    4: frozenset([\'H\', \'N\', \'Q\', \'W\']),
    5: frozenset([\'R\', \'K\']),
    6: frozenset([\'D\', \'E\']),
    7: frozenset([\'C\']),
}
C2_KEYS = (1, 2, 3, 4, 5, 6, 7)

MAX_SEQ_LEN = 1000
MAX_PKT_LEN = 63
MAX_SMI_LEN = 150


def encode_residue(aa, sse=\'C\'):
    feat = np.zeros(40, dtype=np.float32)
    aa = aa if aa in AA_INDEX else \'X\'
    for i, key in enumerate(C1_KEYS):
        feat[i] = (1.0 / len(C1_KEYS)) if aa == \'X\' else (1.0 if aa in C1_GROUPS[key] else 0.0)
    for i, key in enumerate(C2_KEYS):
        feat[4 + i] = (1.0 / len(C2_KEYS)) if aa == \'X\' else (1.0 if aa in C2_GROUPS[key] else 0.0)
    sse_char = sse if sse in SSE_INDEX else \'C\'
    feat[11 + SSE_INDEX[sse_char]] = 1.0
    feat[19 + AA_INDEX[aa]] = 1.0
    return feat


def encode_sequence(aa_seq, sse_seq=\'\', max_len=MAX_SEQ_LEN):
    result = np.zeros((max_len, PT_FEATURE_SIZE), dtype=np.float32)
    for i, aa in enumerate(aa_seq[:max_len]):
        sse = sse_seq[i] if i < len(sse_seq) else \'C\'
        result[i] = encode_residue(aa, sse)
    return result


def label_smiles(smiles, max_len=MAX_SMI_LEN):
    result = np.zeros(max_len, dtype=np.int64)
    for i, ch in enumerate(smiles[:max_len]):
        if ch in CHAR_SMI_SET:
            result[i] = CHAR_SMI_SET[ch] - 1
    return result


class PDBbindDataset(Dataset):
    def __init__(self, processed_dir, split_file):
        self.processed_dir = processed_dir
        with open(split_file) as f:
            self.ids = [line.strip() for line in f if line.strip()]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        pid = self.ids[idx]
        d = self.processed_dir
        seq = torch.from_numpy(np.load(os.path.join(d, f\'{pid}_seq.npy\')))
        pkt = torch.from_numpy(np.load(os.path.join(d, f\'{pid}_pkt.npy\')))
        smi = torch.from_numpy(np.load(os.path.join(d, f\'{pid}_smi.npy\')))
        aff = torch.tensor(float(np.load(os.path.join(d, f\'{pid}_aff.npy\'))), dtype=torch.float32)
        return seq, pkt, smi, aff


class DummyDataset(Dataset):
    def __init__(self, n=64):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        seq = torch.zeros(MAX_SEQ_LEN, PT_FEATURE_SIZE, dtype=torch.float32)
        pkt = torch.zeros(MAX_PKT_LEN, PT_FEATURE_SIZE, dtype=torch.float32)
        smi = torch.zeros(MAX_SMI_LEN, dtype=torch.int64)
        aff = torch.tensor(5.0 + torch.rand(1).item(), dtype=torch.float32)
        return seq, pkt, smi, aff
'''
with open('dataset.py', 'w') as f:
    f.write(dataset_src)
print('dataset.py written')

In [ ]:
# Cell 4 — Write model.py
model_src = '''
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
import metrics
from dataset import PT_FEATURE_SIZE
import torch.nn.functional as F
import pennylane as qml


def GetVQC(n_qubits, qnn_layers, qnn_type):
    if qnn_type == \'ReUploadingVQC\':
        def ReUploadingVQC(inputs, entangling_weights, embedding_weights):
            # default.qubit starts in |0...0> — no BasisStatePreparation needed
            qml.AngleEmbedding(inputs, wires=range(n_qubits))
            for i in range(qnn_layers):
                qml.StronglyEntanglingLayers(entangling_weights[i], wires=range(n_qubits))
                features = inputs * embedding_weights[i]
                qml.AngleEmbedding(features=features, wires=range(n_qubits))
            qml.StronglyEntanglingLayers(entangling_weights[-1], wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

        entangling_weights_shape = (qnn_layers + 1,) + qml.StronglyEntanglingLayers.shape(n_layers=1, n_wires=n_qubits)
        embedding_weights_shape = (qnn_layers, n_qubits)
        weight_shapes = {
            \'entangling_weights\': entangling_weights_shape,
            \'embedding_weights\': embedding_weights_shape,
        }
        return ReUploadingVQC, weight_shapes

    elif qnn_type == \'NormalVQC\':
        def NormalVQC(inputs, entangling_weights):
            qml.AngleEmbedding(features=inputs, wires=range(n_qubits))
            qml.StronglyEntanglingLayers(entangling_weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

        entangling_weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=qnn_layers, n_wires=n_qubits)
        weight_shapes = {\'entangling_weights\': entangling_weights_shape}
        return NormalVQC, weight_shapes


qnn_type = \'ReUploadingVQC\'
n_qubits = 10
qnn_layers = 20
CHAR_SMI_SET_LEN = 64

counter = 0


class Squeeze(nn.Module):
    def forward(self, input):
        return input.squeeze()


class CDilated(nn.Module):
    def __init__(self, nIn, nOut, kSize, stride=1, d=1):
        super().__init__()
        padding = int((kSize - 1) / 2) * d
        self.conv = nn.Conv1d(nIn, nOut, kSize, stride=stride, padding=padding, bias=False, dilation=d)

    def forward(self, input):
        global counter
        output = self.conv(input)
        counter += 1
        return output


class DilatedParllelResidualBlockA(nn.Module):
    def __init__(self, nIn, nOut, add=True):
        super().__init__()
        n = int(nOut / 5)
        n1 = nOut - 4 * n
        self.c1 = nn.Conv1d(nIn, n, 1, padding=0)
        self.br1 = nn.Sequential(nn.BatchNorm1d(n), nn.PReLU())
        self.d1 = CDilated(n, n1, 3, 1, 1)
        self.d2 = CDilated(n, n, 3, 1, 2)
        self.d4 = CDilated(n, n, 3, 1, 4)
        self.d8 = CDilated(n, n, 3, 1, 8)
        self.d16 = CDilated(n, n, 3, 1, 16)
        self.br2 = nn.Sequential(nn.BatchNorm1d(nOut), nn.PReLU())
        if nIn != nOut:
            add = False
        self.add = add

    def forward(self, input):
        output1 = self.br1(self.c1(input))
        d1 = self.d1(output1)
        d2 = self.d2(output1)
        d4 = self.d4(output1)
        d8 = self.d8(output1)
        d16 = self.d16(output1)
        add1 = d2
        add2 = add1 + d4
        add3 = add2 + d8
        add4 = add3 + d16
        combine = torch.cat([d1, add1, add2, add3, add4], 1)
        if self.add:
            combine = input + combine
        return self.br2(combine)


class DilatedParllelResidualBlockB(nn.Module):
    def __init__(self, nIn, nOut, add=True):
        super().__init__()
        n = int(nOut / 4)
        n1 = nOut - 3 * n
        self.c1 = nn.Conv1d(nIn, n, 1, padding=0)
        self.br1 = nn.Sequential(nn.BatchNorm1d(n), nn.PReLU())
        self.d1 = CDilated(n, n1, 3, 1, 1)
        self.d2 = CDilated(n, n, 3, 1, 2)
        self.d4 = CDilated(n, n, 3, 1, 4)
        self.d8 = CDilated(n, n, 3, 1, 8)
        self.br2 = nn.Sequential(nn.BatchNorm1d(nOut), nn.PReLU())
        if nIn != nOut:
            add = False
        self.add = add

    def forward(self, input):
        output1 = self.br1(self.c1(input))
        d1 = self.d1(output1)
        d2 = self.d2(output1)
        d4 = self.d4(output1)
        d8 = self.d8(output1)
        add1 = d2
        add2 = add1 + d4
        add3 = add2 + d8
        combine = torch.cat([d1, add1, add2, add3], 1)
        if self.add:
            combine = input + combine
        return self.br2(combine)


class DeepDTAF(nn.Module):
    def __init__(self):
        super().__init__()

        smi_embed_size = 128
        seq_embed_size = 128
        seq_oc = 128
        pkt_oc = 128
        smi_oc = 128

        self.smi_embed = nn.Embedding(CHAR_SMI_SET_LEN, smi_embed_size)
        self.seq_embed = nn.Linear(PT_FEATURE_SIZE, seq_embed_size)

        dev = qml.device(\'default.qubit\', wires=n_qubits)
        VQC, weight_shapes = GetVQC(n_qubits, qnn_layers, qnn_type)
        qnode = qml.QNode(VQC, dev, interface=\'torch\', diff_method=\'best\')
        self.qlayerC = qml.qnn.TorchLayer(qnode, weight_shapes)

        conv_seq = []
        ic = seq_embed_size
        for oc in [32, 64, 64, seq_oc]:
            conv_seq.append(DilatedParllelResidualBlockA(ic, oc))
            ic = oc
        conv_seq.append(nn.AdaptiveMaxPool1d(1))
        conv_seq.append(Squeeze())
        self.conv_seq = nn.Sequential(*conv_seq)

        conv_pkt = []
        ic = seq_embed_size
        for oc in [32, 64, pkt_oc]:
            conv_pkt.append(nn.Conv1d(ic, oc, 3))
            conv_pkt.append(nn.BatchNorm1d(oc))
            conv_pkt.append(nn.PReLU())
            ic = oc
        conv_pkt.append(nn.AdaptiveMaxPool1d(1))
        conv_pkt.append(Squeeze())
        self.conv_pkt = nn.Sequential(*conv_pkt)

        conv_smi = []
        ic = smi_embed_size
        for oc in [32, 64, smi_oc]:
            conv_smi.append(DilatedParllelResidualBlockB(ic, oc))
            ic = oc
        conv_smi.append(nn.AdaptiveMaxPool1d(1))
        conv_smi.append(Squeeze())
        self.conv_smi = nn.Sequential(*conv_smi)

        self.cat_dropout = nn.Dropout(0.2)
        self.clf = nn.Sequential(nn.Linear(n_qubits, 1), nn.PReLU())
        self.clangle = nn.Linear(384, n_qubits)

    def forward(self, seq, pkt, smi):
        seq_embed = torch.transpose(self.seq_embed(seq), 1, 2)
        seq_conv = self.conv_seq(seq_embed)
        pkt_embed = torch.transpose(self.seq_embed(pkt), 1, 2)
        pkt_conv = self.conv_pkt(pkt_embed)
        smi_embed = torch.transpose(self.smi_embed(smi), 1, 2)
        smi_conv = self.conv_smi(smi_embed)
        cat = self.cat_dropout(torch.cat([seq_conv, pkt_conv, smi_conv], dim=1))
        output_ = self.clangle(cat)
        qout = self.qlayerC(output_)
        return self.clf(qout)


def test(model, test_loader, loss_function, device, show=True):
    model.eval()
    test_loss = 0
    outputs = []
    targets = []
    with torch.no_grad():
        for idx, (*x, y) in tqdm(enumerate(test_loader), disable=not show, total=len(test_loader)):
            for i in range(len(x)):
                x[i] = x[i].to(device)
            y = y.to(device)
            y_hat = model(*x)
            test_loss += loss_function(y_hat.view(-1), y.view(-1)).item()
            outputs.append(y_hat.cpu().numpy().reshape(-1))
            targets.append(y.cpu().numpy().reshape(-1))

    targets = np.concatenate(targets).reshape(-1)
    outputs = np.concatenate(outputs).reshape(-1)
    test_loss /= len(test_loader.dataset)

    return {
        \'loss\': test_loss,
        \'c_index\': metrics.c_index(targets, outputs),
        \'RMSE\': metrics.RMSE(targets, outputs),
        \'MAE\': metrics.MAE(targets, outputs),
        \'SD\': metrics.SD(targets, outputs),
        \'CORR\': metrics.CORR(targets, outputs),
    }
'''
with open('model.py', 'w') as f:
    f.write(model_src)
print('model.py written')

In [ ]:
# Cell 5 — Run the smoke test (2 epochs, 1 run, dummy data)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Reload modules (important after writing files in this session)
import importlib, sys
for mod in ['metrics', 'dataset', 'model']:
    if mod in sys.modules:
        del sys.modules[mod]

from model import DeepDTAF, test
from dataset import DummyDataset

EPOCHS = 2
BATCH_SIZE = 16
LR = 0.005
WEIGHT_DECAY = 0.01

device = torch.device('cpu')

print('Building DummyDataset (64 train / 16 test)...')
train_ds = DummyDataset(n=64)
test_ds  = DummyDataset(n=16)

# drop_last=True avoids the Squeeze() batch-size-1 edge case
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=True)

print('Instantiating DeepDTAF (this compiles the quantum circuit — may take ~30s)...')
model = DeepDTAF().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()

print(f'\nStarting smoke test: {EPOCHS} epochs, batch_size={BATCH_SIZE}')
print('NOTE: Each batch runs the quantum circuit — expect ~1-3 min per epoch on Colab CPU.\n')

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    n_samples = 0
    for seq, pkt, smi, label in train_loader:
        seq, pkt, smi, label = seq.to(device), pkt.to(device), smi.to(device), label.to(device)
        pred = model(seq, pkt, smi)
        loss = loss_fn(pred.view(-1), label.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(label)
        n_samples += len(label)
    train_loss = epoch_loss / n_samples
    print(f'Epoch {epoch}/{EPOCHS}  train_MSE={train_loss:.4f}')

print('\nRunning test evaluation...')
results = test(model, test_loader, loss_fn, device, show=True)

print('\n========== Smoke Test Results ==========')
for k, v in results.items():
    print(f'  {k:10s}: {v:.4f}')
print('\nSmoke test PASSED — model runs end-to-end.')